In [1]:
# Cell 0: Imports & constants
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ccxt
import time
import re
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

OI_SNAPSHOTS = Path("/home/ubuntu/dev/gmx-data-collector/user_data/data/gmx/open_interest/arbitrum/snapshots")
POOL_SNAPSHOTS = Path("/home/ubuntu/dev/gmx-data-collector/user_data/data/gmx/pool_liquidity/arbitrum/snapshots")
BINANCE_CACHE = Path("/home/ubuntu/dev/gmx-ccxt-freqtrade/notebooks/analysis/binance_daily_cache.parquet")
USDC_TOKEN = "0xaf88d065e77c8cc2239327c5edb3a432268e5831"

# Alt-collateral pattern: dirs like BTC_USD_WBTC.b-WBTC.b (has 3+ underscore-separated segments)
ALT_COLLATERAL_RE = re.compile(r"^[A-Z0-9.]+_USD_.+$")

ROLLING_WINDOWS = [7, 14, 30]
TOP_N_VALUES = [5, 7, 10, 15, 20]

print("Setup complete.")

Setup complete.


In [2]:
# Cell 1: Load GMX OI daily snapshots
oi_frames = []
for d in sorted(OI_SNAPSHOTS.iterdir()):
    if not d.is_dir() or ALT_COLLATERAL_RE.match(d.name):
        continue
    parquet = d / "daily.parquet"
    if not parquet.exists():
        continue
    df = pd.read_parquet(parquet)
    oi_frames.append(df)

oi_daily = pd.concat(oi_frames, ignore_index=True)
oi_daily["date"] = pd.to_datetime(oi_daily["date"])
oi_daily["totalOiUsd"] = pd.to_numeric(oi_daily["totalOiUsd"], errors="coerce")

# Normalize symbol: "BTC/USD" -> "BTC"
oi_daily["base"] = oi_daily["symbol"].str.replace("/USD", "", regex=False)

print(f"OI data: {len(oi_daily):,} rows, {oi_daily['base'].nunique()} markets")
print(f"Date range: {oi_daily['date'].min().date()} to {oi_daily['date'].max().date()}")
oi_daily[["date", "base", "totalOiUsd"]].head()

OI data: 41,222 rows, 109 markets
Date range: 2023-08-10 to 2026-03-11


,date,base,totalOiUsd
0,2025-10-09,0G,120.32
1,2025-10-10,0G,116.23
2,2025-10-11,0G,1.03
3,2025-10-13,0G,50.94
4,2025-10-17,0G,1.03


In [3]:
# Cell 2: Load GMX pool liquidity daily snapshots
pool_frames = []
for d in sorted(POOL_SNAPSHOTS.iterdir()):
    if not d.is_dir() or ALT_COLLATERAL_RE.match(d.name):
        continue
    parquet = d / "daily.parquet"
    if not parquet.exists():
        continue
    df = pd.read_parquet(parquet)
    pool_frames.append(df)

pool_raw = pd.concat(pool_frames, ignore_index=True)
pool_raw["date"] = pd.to_datetime(pool_raw["date"]).dt.tz_localize(None)
pool_raw["base"] = pool_raw["symbol"].str.replace("/USD", "", regex=False)

# Pivot: separate USDC pool vs native token pool
pool_raw["is_usdc"] = pool_raw["token"].str.lower() == USDC_TOKEN

usdc_pool = (
    pool_raw[pool_raw["is_usdc"]]
    .groupby(["date", "base"])["pool_tokens"]
    .sum()
    .rename("usdc_pool")
)
native_pool = (
    pool_raw[~pool_raw["is_usdc"]]
    .groupby(["date", "base"])["pool_tokens"]
    .sum()
    .rename("native_pool")
)

pool_daily = pd.concat([usdc_pool, native_pool], axis=1).reset_index().fillna(0)

print(f"Pool liquidity: {len(pool_daily):,} rows, {pool_daily['base'].nunique()} markets")
pool_daily.head()

Pool liquidity: 42,236 rows, 109 markets


,date,base,usdc_pool,native_pool
0,2023-08-10,ARB,4.528020e+05,417872.030257
1,2023-08-10,BTC,1.681528e+06,94.546692
2,2023-08-10,DOGE,2.125070e+04,62.486009
3,2023-08-10,ETH,3.940795e+06,2137.215551
4,2023-08-10,LINK,9.240896e+05,109863.988688


In [4]:
# Cell 3: Build GMX→Binance symbol mapping & fetch Binance daily OHLCV
exchange = ccxt.binanceusdm({"enableRateLimit": True})
exchange.load_markets()

gmx_bases = sorted(oi_daily["base"].unique())
binance_map = {}  # base -> binance symbol
gmx_only = []     # bases not on Binance

for base in gmx_bases:
    # Try standard futures symbol
    candidates = [f"{base}/USDT:USDT"]
    # Handle special cases
    if base == "XAUT.v2" or base == "XAUT (deprecated)":
        candidates = ["PAXG/USDT:USDT"]
    elif base == "S":
        candidates = ["S/USDT:USDT", "SONIC/USDT:USDT"]
    elif base == "POL":
        candidates = ["POL/USDT:USDT", "MATIC/USDT:USDT"]

    found = False
    for sym in candidates:
        if sym in exchange.markets:
            binance_map[base] = sym
            found = True
            break
    if not found:
        gmx_only.append(base)

print(f"Mapped to Binance: {len(binance_map)} symbols")
print(f"GMX-only (no Binance match): {len(gmx_only)} -> {gmx_only}")

Mapped to Binance: 97 symbols
GMX-only (no Binance match): 12 -> ['BONK', 'CRO', 'FLOKI', 'KTA', 'MNT', 'OKB', 'PEPE', 'PI', 'SATS', 'SHIB', 'SPX6900', 'WELL']


In [5]:
# Cell 3b: Fetch Binance daily OHLCV (cached)
if BINANCE_CACHE.exists():
    binance_daily = pd.read_parquet(BINANCE_CACHE)
    print(f"Loaded cached Binance data: {len(binance_daily):,} rows")
else:
    since = int(pd.Timestamp("2023-08-01").timestamp() * 1000)
    rows = []
    for i, (base, bsym) in enumerate(binance_map.items()):
        try:
            ohlcv = exchange.fetch_ohlcv(bsym, "1d", since=since, limit=1000)
            for candle in ohlcv:
                rows.append({
                    "base": base,
                    "date": pd.Timestamp(candle[0], unit="ms").normalize(),
                    "close": candle[4],
                    "volume_usd": candle[4] * candle[5],  # close * base_volume
                })
            if (i + 1) % 20 == 0:
                print(f"  Fetched {i+1}/{len(binance_map)}...")
        except Exception as e:
            print(f"  Skip {base} ({bsym}): {e}")
        time.sleep(exchange.rateLimit / 1000)

    binance_daily = pd.DataFrame(rows)
    binance_daily.to_parquet(BINANCE_CACHE)
    print(f"Fetched & cached Binance data: {len(binance_daily):,} rows")

print(f"Binance symbols: {binance_daily['base'].nunique()}")
print(f"Date range: {binance_daily['date'].min().date()} to {binance_daily['date'].max().date()}")

  Fetched 20/97...


  Fetched 40/97...


  Fetched 60/97...


  Fetched 80/97...


Fetched & cached Binance data: 66,268 rows
Binance symbols: 97
Date range: 2023-08-01 to 2026-03-11


In [6]:
# Cell 4: Merge all three datasets
merged = (
    oi_daily[["date", "base", "totalOiUsd"]]
    .merge(pool_daily[["date", "base", "usdc_pool", "native_pool"]], on=["date", "base"], how="inner")
    .merge(binance_daily[["date", "base", "close", "volume_usd"]], on=["date", "base"], how="inner")
)

# Compute total pool USD value
merged["pool_usd"] = merged["usdc_pool"] + merged["native_pool"] * merged["close"]

# Drop rows with missing/zero OI or volume
merged = merged[(merged["totalOiUsd"] > 0) & (merged["volume_usd"] > 0)].copy()

print(f"Merged dataset: {len(merged):,} rows, {merged['base'].nunique()} overlapping symbols")
print(f"Date range: {merged['date'].min().date()} to {merged['date'].max().date()}")
print(f"\nTop 10 by latest OI:")
latest = merged[merged["date"] == merged["date"].max()].nlargest(10, "totalOiUsd")
latest[["base", "totalOiUsd", "volume_usd", "pool_usd"]].to_string(index=False)

Merged dataset: 37,539 rows, 97 overlapping symbols
Date range: 2023-08-10 to 2026-03-11

Top 10 by latest OI:


'   base  totalOiUsd   volume_usd     pool_usd\n    ETH 34927041.58 7.199783e+09 6.152956e+07\n    BTC 23953800.91 1.048498e+10 8.008969e+07\n   LINK 12883895.47 9.281372e+07 6.558744e+06\n    XRP  5937184.72 4.996904e+08 3.864168e+06\n    SOL  2115673.59 1.589329e+09 6.875058e+06\n   HYPE   768300.93 3.639501e+08 1.032819e+06\nXAUT.v2   520226.26 8.634640e+07 8.276975e+05\n   AVAX   420222.09 1.056271e+08 2.355775e+05\n    XMR   380987.21 2.416289e+07 6.747279e+05\n    SUI   370882.38 1.470254e+08 4.848503e+05'

In [7]:
# Cell 5: Compute rolling averages
merged = merged.sort_values(["base", "date"])

for w in ROLLING_WINDOWS:
    merged[f"rolling_oi_{w}d"] = merged.groupby("base")["totalOiUsd"].transform(
        lambda x: x.rolling(w, min_periods=max(1, w // 2)).mean()
    )
    merged[f"rolling_vol_{w}d"] = merged.groupby("base")["volume_usd"].transform(
        lambda x: x.rolling(w, min_periods=max(1, w // 2)).mean()
    )
    merged[f"rolling_pool_{w}d"] = merged.groupby("base")["pool_usd"].transform(
        lambda x: x.rolling(w, min_periods=max(1, w // 2)).mean()
    )

print("Rolling averages computed for windows:", ROLLING_WINDOWS)

Rolling averages computed for windows: [7, 14, 30]


In [8]:
# Cell 6: Daily rank computation
for w in ROLLING_WINDOWS:
    merged[f"rank_vol_{w}d"] = merged.groupby("date")[f"rolling_vol_{w}d"].rank(
        ascending=False, method="min"
    )
    merged[f"rank_oi_{w}d"] = merged.groupby("date")[f"rolling_oi_{w}d"].rank(
        ascending=False, method="min"
    )

print("Daily ranks computed.")
# Show a sample: latest date, 14d window
sample = merged[merged["date"] == merged["date"].max()].nlargest(15, "rolling_oi_14d")
sample[["base", "rank_vol_14d", "rank_oi_14d", "rolling_vol_14d", "rolling_oi_14d"]].reset_index(drop=True)

Daily ranks computed.


,base,rank_vol_14d,rank_oi_14d,rolling_vol_14d,rolling_oi_14d
0,ETH,2.0,1.0,1.135909e+10,3.108672e+07
1,BTC,1.0,2.0,1.516407e+10,2.262953e+07
2,LINK,17.0,3.0,1.308821e+08,3.916378e+06
3,SOL,3.0,4.0,2.431889e+09,2.596279e+06
4,XRP,4.0,5.0,9.431287e+08,1.410532e+06
5,HYPE,7.0,6.0,4.087348e+08,8.033344e+05
6,SUI,10.0,7.0,2.264693e+08,5.069349e+05
7,XPL,25.0,8.0,6.205763e+07,4.463324e+05
8,XAUT.v2,9.0,9.0,2.931968e+08,4.381768e+05
9,AVAX,14.0,10.0,1.579471e+08,3.784745e+05


In [9]:
# Cell 7: Spearman rank correlation over time
def daily_spearman(group):
    """Compute Spearman rho between volume rank and OI rank for one day."""
    results = {}
    for w in ROLLING_WINDOWS:
        vol_col = f"rank_vol_{w}d"
        oi_col = f"rank_oi_{w}d"
        valid = group[[vol_col, oi_col]].dropna()
        if len(valid) >= 5:
            rho, _ = spearmanr(valid[vol_col], valid[oi_col])
            results[f"spearman_{w}d"] = rho
        else:
            results[f"spearman_{w}d"] = np.nan
    return pd.Series(results)

spearman_ts = merged.groupby("date").apply(daily_spearman, include_groups=False).reset_index()

fig = go.Figure()
colors = {"7": "#636EFA", "14": "#EF553B", "30": "#00CC96"}
for w in ROLLING_WINDOWS:
    col = f"spearman_{w}d"
    # Smooth with 7d rolling for readability
    smoothed = spearman_ts[col].rolling(7, min_periods=1).mean()
    fig.add_trace(go.Scatter(
        x=spearman_ts["date"], y=smoothed,
        name=f"{w}d window", line=dict(color=colors[str(w)]),
    ))

fig.add_hline(y=0.7, line_dash="dash", line_color="white", opacity=0.5,
              annotation_text="Strong proxy (ρ=0.7)")
fig.update_layout(
    title="Spearman Rank Correlation: Volume Rank vs OI Rank (7d smoothed)",
    yaxis_title="Spearman ρ", xaxis_title="Date",
    template="plotly_dark", height=500,
    legend=dict(x=0.02, y=0.98),
)
fig.show()

# Summary stats
for w in ROLLING_WINDOWS:
    col = f"spearman_{w}d"
    vals = spearman_ts[col].dropna()
    print(f"  {w}d window: mean ρ = {vals.mean():.3f}, median = {vals.median():.3f}, "
          f"std = {vals.std():.3f}, % days > 0.7 = {(vals > 0.7).mean()*100:.1f}%")

  7d window: mean ρ = 0.579, median = 0.589, std = 0.139, % days > 0.7 = 13.1%
  14d window: mean ρ = 0.583, median = 0.596, std = 0.137, % days > 0.7 = 13.3%
  30d window: mean ρ = 0.594, median = 0.600, std = 0.133, % days > 0.7 = 15.3%


In [10]:
# Cell 8: Top-N Jaccard similarity over time
def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

jaccard_results = []
w = 14  # Primary rolling window
for date, group in merged.groupby("date"):
    valid = group.dropna(subset=[f"rolling_vol_{w}d", f"rolling_oi_{w}d"])
    if len(valid) < 5:
        continue
    for n in TOP_N_VALUES:
        top_vol = set(valid.nlargest(n, f"rolling_vol_{w}d")["base"])
        top_oi = set(valid.nlargest(n, f"rolling_oi_{w}d")["base"])
        jaccard_results.append({
            "date": date,
            "n": n,
            "jaccard": jaccard(top_vol, top_oi),
            "overlap": len(top_vol & top_oi),
        })

jaccard_df = pd.DataFrame(jaccard_results)

fig = px.line(
    jaccard_df, x="date", y="jaccard", color="n",
    title="Top-N Jaccard Similarity: Volume vs OI (14d rolling)",
    labels={"jaccard": "Jaccard Similarity", "n": "Top-N"},
    template="plotly_dark", height=500,
)
fig.add_hline(y=0.6, line_dash="dash", line_color="white", opacity=0.5,
              annotation_text="Good overlap (J=0.6)")
fig.show()

# Summary
for n in TOP_N_VALUES:
    subset = jaccard_df[jaccard_df["n"] == n]
    avg_j = subset["jaccard"].mean()
    avg_overlap = subset["overlap"].mean()
    print(f"  Top-{n}: mean Jaccard = {avg_j:.3f}, mean overlap = {avg_overlap:.1f}/{n} coins")

  Top-5: mean Jaccard = 0.520, mean overlap = 3.4/5 coins
  Top-7: mean Jaccard = 0.534, mean overlap = 4.8/7 coins
  Top-10: mean Jaccard = 0.615, mean overlap = 7.2/10 coins
  Top-15: mean Jaccard = 0.692, mean overlap = 10.6/15 coins
  Top-20: mean Jaccard = 0.733, mean overlap = 13.2/20 coins


In [11]:
# Cell 9: Liquidity filter test — does removing low-liquidity coins improve alignment?
w = 14
thresholds = {"No filter": 0, "Remove bottom 10%": 10, "Remove bottom 25%": 25, "Remove bottom 50%": 50}

filter_results = []
for date, group in merged.groupby("date"):
    valid = group.dropna(subset=[f"rolling_vol_{w}d", f"rolling_oi_{w}d", f"rolling_pool_{w}d"])
    if len(valid) < 10:
        continue
    for label, pctile in thresholds.items():
        if pctile > 0:
            cutoff = valid[f"rolling_pool_{w}d"].quantile(pctile / 100)
            filtered = valid[valid[f"rolling_pool_{w}d"] >= cutoff]
        else:
            filtered = valid
        if len(filtered) < 5:
            continue
        for n in [7, 10]:
            top_vol = set(filtered.nlargest(n, f"rolling_vol_{w}d")["base"])
            top_oi = set(filtered.nlargest(n, f"rolling_oi_{w}d")["base"])
            filter_results.append({
                "date": date, "filter": label, "n": n,
                "jaccard": jaccard(top_vol, top_oi),
            })

filter_df = pd.DataFrame(filter_results)

# Plot for top-7
fig = px.line(
    filter_df[filter_df["n"] == 7],
    x="date", y="jaccard", color="filter",
    title="Effect of Liquidity Filtering on Top-7 OI vs Volume Overlap (14d rolling)",
    labels={"jaccard": "Jaccard Similarity", "filter": "Liquidity Filter"},
    template="plotly_dark", height=500,
)
fig.show()

# Summary table
print("\nAverage Jaccard by filter threshold:")
for n in [7, 10]:
    print(f"\n  Top-{n}:")
    for label in thresholds:
        subset = filter_df[(filter_df["n"] == n) & (filter_df["filter"] == label)]
        print(f"    {label:>20s}: Jaccard = {subset['jaccard'].mean():.3f}")


Average Jaccard by filter threshold:

  Top-7:
               No filter: Jaccard = 0.507
       Remove bottom 10%: Jaccard = 0.509
       Remove bottom 25%: Jaccard = 0.537
       Remove bottom 50%: Jaccard = 0.623

  Top-10:
               No filter: Jaccard = 0.531
       Remove bottom 10%: Jaccard = 0.547
       Remove bottom 25%: Jaccard = 0.586
       Remove bottom 50%: Jaccard = 0.687


In [12]:
# Cell 10: Membership stability — per-coin frequency in top-7
w = 14
n = 7

membership = []
for date, group in merged.groupby("date"):
    valid = group.dropna(subset=[f"rolling_vol_{w}d", f"rolling_oi_{w}d"])
    if len(valid) < n:
        continue
    top_vol = set(valid.nlargest(n, f"rolling_vol_{w}d")["base"])
    top_oi = set(valid.nlargest(n, f"rolling_oi_{w}d")["base"])
    for base in valid["base"].unique():
        membership.append({
            "base": base, "date": date,
            "in_vol_top7": base in top_vol,
            "in_oi_top7": base in top_oi,
        })

mem_df = pd.DataFrame(membership)
freq = mem_df.groupby("base").agg(
    pct_vol=("in_vol_top7", "mean"),
    pct_oi=("in_oi_top7", "mean"),
    days=("date", "count"),
).reset_index()
freq = freq[freq["days"] >= 30]  # at least 30 days of data

fig = px.scatter(
    freq, x="pct_vol", y="pct_oi", text="base",
    title="Top-7 Membership Stability: Volume vs OI (14d rolling)",
    labels={"pct_vol": "% Days in Volume Top-7", "pct_oi": "% Days in OI Top-7"},
    template="plotly_dark", height=600, width=800,
)
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1,
              line=dict(color="white", dash="dash", width=1))
fig.update_traces(textposition="top center", textfont_size=9)
fig.show()

# Categorize
always_both = freq[(freq["pct_vol"] > 0.5) & (freq["pct_oi"] > 0.5)]
vol_only = freq[(freq["pct_vol"] > 0.3) & (freq["pct_oi"] < 0.1)]
oi_only = freq[(freq["pct_oi"] > 0.3) & (freq["pct_vol"] < 0.1)]

print(f"\nAlways in both top-7 (>50% of days): {sorted(always_both['base'].tolist())}")
print(f"Volume-only (>30% vol, <10% OI): {sorted(vol_only['base'].tolist())}")
print(f"OI-only (>30% OI, <10% vol): {sorted(oi_only['base'].tolist())}")


Always in both top-7 (>50% of days): ['BTC', 'DOGE', 'ETH', 'SOL', 'XRP']
Volume-only (>30% vol, <10% OI): ['BNB', 'ZEC']
OI-only (>30% OI, <10% vol): ['GMX']


In [13]:
# Cell 11: Side-by-side top-7 comparison at sample dates
w = 14
n = 7

# Pick dates every ~90 days
all_dates = sorted(merged["date"].unique())
sample_dates = [all_dates[i] for i in range(0, len(all_dates), 90)] + [all_dates[-1]]

rows = []
for date in sample_dates:
    group = merged[merged["date"] == date].dropna(subset=[f"rolling_vol_{w}d", f"rolling_oi_{w}d"])
    if len(group) < n:
        continue
    top_vol = list(group.nlargest(n, f"rolling_vol_{w}d")["base"])
    top_oi = list(group.nlargest(n, f"rolling_oi_{w}d")["base"])
    overlap = set(top_vol) & set(top_oi)
    rows.append({
        "Date": pd.Timestamp(date).strftime("%Y-%m-%d"),
        "Volume Top-7": ", ".join(top_vol),
        "OI Top-7": ", ".join(top_oi),
        "Overlap": f"{len(overlap)}/7 ({', '.join(sorted(overlap))})",
    })

comparison_df = pd.DataFrame(rows)
comparison_df.style.set_properties(**{"text-align": "left"})

,Date,Volume Top-7,OI Top-7,Overlap
0,2023-11-08,"BTC, ETH, SOL, XRP, LINK, DOGE, ARB","BTC, ETH, ARB, LINK, SOL, UNI, DOGE","6/7 (ARB, BTC, DOGE, ETH, LINK, SOL)"
1,2024-02-06,"BTC, ETH, SOL, LINK, XRP, ARB, BNB","BTC, ETH, SOL, ARB, LINK, XRP, DOGE","6/7 (ARB, BTC, ETH, LINK, SOL, XRP)"
2,2024-05-06,"BTC, ETH, SOL, DOGE, BNB, XRP, NEAR","ETH, BTC, SOL, LINK, ARB, DOGE, NEAR","5/7 (BTC, DOGE, ETH, NEAR, SOL)"
3,2024-08-04,"BTC, ETH, SOL, XRP, WIF, DOGE, BNB","ETH, BTC, SOL, LINK, ARB, DOGE, GMX","4/7 (BTC, DOGE, ETH, SOL)"
4,2024-11-02,"BTC, ETH, SOL, DOGE, SUI, WIF, APE","BTC, ETH, SOL, LINK, DOGE, GMX, WIF","5/7 (BTC, DOGE, ETH, SOL, WIF)"
5,2025-01-31,"BTC, ETH, SOL, TRUMP, XRP, DOGE, SUI","BTC, ETH, SOL, LINK, DOGE, GMX, XRP","5/7 (BTC, DOGE, ETH, SOL, XRP)"
6,2025-05-01,"BTC, ETH, SOL, XRP, SUI, TRUMP, DOGE","BTC, ETH, LINK, SOL, GMX, XRP, FARTCOIN","4/7 (BTC, ETH, SOL, XRP)"
7,2025-07-30,"ETH, BTC, SOL, XRP, DOGE, SUI, ENA","BTC, ETH, LINK, DOGE, XRP, SOL, SUI","6/7 (BTC, DOGE, ETH, SOL, SUI, XRP)"
8,2025-10-28,"ETH, BTC, SOL, BNB, XRP, DOGE, ASTER","ETH, BTC, LINK, SOL, XRP, HYPE, DOGE","5/7 (BTC, DOGE, ETH, SOL, XRP)"
9,2026-01-26,"BTC, ETH, SOL, XRP, DASH, ZEC, DOGE","ETH, BTC, LINK, SOL, SUI, XMR, DOGE","4/7 (BTC, DOGE, ETH, SOL)"


In [14]:
# Cell 12: Rank difference heatmap (top 30 symbols by average volume)
w = 14
top30 = merged.groupby("base")[f"rolling_vol_{w}d"].mean().nlargest(30).index.tolist()
heatmap_data = merged[merged["base"].isin(top30)].copy()
heatmap_data["rank_diff"] = (heatmap_data[f"rank_vol_{w}d"] - heatmap_data[f"rank_oi_{w}d"]).abs()

pivot = heatmap_data.pivot_table(index="base", columns="date", values="rank_diff")
# Resample to weekly for readability
pivot = pivot.T.resample("W").mean().T

fig = px.imshow(
    pivot.values,
    x=pivot.columns.strftime("%Y-%m-%d"),
    y=pivot.index,
    color_continuous_scale="RdYlGn_r",
    labels={"color": "|Rank Diff|"},
    title="Absolute Rank Difference: Volume vs OI (top 30 by avg volume, weekly avg)",
    template="plotly_dark",
    height=700,
    aspect="auto",
)
fig.update_xaxes(dtick="M3", tickformat="%b %Y")
fig.show()

In [15]:
# Cell 13: Pool utilization (OI / pool_usd) vs volume rank
w = 14
latest_window = merged["date"].max() - pd.Timedelta(days=30)
recent = merged[merged["date"] >= latest_window].copy()

avg_recent = recent.groupby("base").agg(
    avg_oi=("totalOiUsd", "mean"),
    avg_pool=("pool_usd", "mean"),
    avg_vol=(f"rolling_vol_{w}d", "mean"),
    avg_vol_rank=(f"rank_vol_{w}d", "mean"),
    avg_oi_rank=(f"rank_oi_{w}d", "mean"),
).reset_index()
avg_recent["utilization"] = avg_recent["avg_oi"] / avg_recent["avg_pool"].replace(0, np.nan)
avg_recent = avg_recent.dropna(subset=["utilization"])

fig = px.scatter(
    avg_recent, x="utilization", y="avg_vol_rank",
    size="avg_oi", text="base", color="avg_oi_rank",
    color_continuous_scale="Viridis_r",
    title="Pool Utilization (OI/Pool USD) vs Volume Rank (last 30 days avg)",
    labels={
        "utilization": "OI / Pool USD (Utilization)",
        "avg_vol_rank": "Avg Volume Rank (lower = higher volume)",
        "avg_oi_rank": "Avg OI Rank",
    },
    template="plotly_dark", height=600, width=900,
)
fig.update_traces(textposition="top center", textfont_size=8)
fig.update_yaxes(autorange="reversed")  # rank 1 at top
fig.show()

## Conclusions

### Key Metrics Summary

| Metric | Value | Interpretation |
|--------|-------|----------------|
| **Spearman ρ (14d)** | See Cell 7 output | > 0.7 = strong proxy, 0.5-0.7 = moderate, < 0.5 = weak |
| **Jaccard Top-7 (14d)** | See Cell 8 output | > 0.6 = good overlap, 0.3-0.6 = partial, < 0.3 = poor |
| **Liquidity filter effect** | See Cell 9 output | Positive delta = filtering helps alignment |

### Interpretation Guide

- **If Spearman ρ consistently > 0.7 and Jaccard > 0.5:** OI is a strong proxy for volume. Sorting by OI with a pool liquidity floor is a viable replacement for VolumePairList.
- **If Spearman ρ = 0.5-0.7:** OI captures the major coins (BTC, ETH, SOL) but diverges on mid-cap. Still usable — the top coins are what matter most for strategy performance.
- **If Spearman ρ < 0.5:** OI and volume measure fundamentally different things. OI-based selection would produce a meaningfully different trading universe.

### Suggested GMX Pair Selection Config

Based on the analysis above, a reasonable approach for GMX:

```
Sort by: 14-day rolling average OI (descending)
Filter: Remove coins with pool liquidity below 25th percentile
Select: Top 7 coins
Rebalance: Daily
```

This is the GMX-native equivalent of:
```
VolumePairList(number_assets=7, sort_key=quoteVolume, lookback_days=14)
```

**Note:** Even if OI doesn't perfectly match volume rankings, it may actually be *better* for GMX because it directly measures what matters on the platform — how much capital is deployed in positions, which drives fees, funding rates, and liquidity depth.

# Binance Volume vs GMX OI + Liquidity Correlation

**Goal:** Determine if ranking by Open Interest and filtering by pool liquidity
is a viable replacement for volume-based pair selection (VolumePairList) on GMX.